# BABA NN

In [5]:
import numpy as np
import tensorflow as tf
print(tf.__version__) 
from matplotlib import pyplot as plt
import pandas as pd
#from PIL import Image

2.21.0


In [6]:
DATA_DIR = "../data"
DATA_TRAIN_DIR = DATA_DIR + "/train"
DATA_CRIT_DIR = DATA_DIR + "/critical"
CSV_PATH = DATA_DIR + "/dataset.csv"

BATCH_SIZE = 32
IMG_SIZE = (10, 10)

### weird dataset loading

In [ ]:
df = pd.read_csv(CSV_PATH)
#train_df = df[df["split"] == "train"]
#val_df = df[df["split"] == "critical"] 

def parse_sample(filepath, label):
    full_path = DATA_DIR + "/" + filepath.numpy().decode()
    img = tf.io.read_file(full_path)
    img = tf.image.decode_png(img, channels=1)        # grayscale; use channels=3 for RGB
    #img = tf.image.resize(img, (L,L))
    img = tf.cast(img, tf.float32) / 255.0            # normalise to [0, 1]
    return img, label

def tf_parse(filepath, label):
    img, label = tf.py_function(
        parse_sample, [filepath, label], [tf.float32, tf.int32]
    )
    img.set_shape([*IMG_SIZE, 1])                  # needed for model to infer input shape
    label.set_shape([])
    return img, label

def make_dataset(df, shuffle=False):
    filepaths = df["filepath"].values
    labels    = df["label"].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=42)
    ds = ds.map(tf_parse, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

"""
ds = make_dataset(df, shuffle=True)

for imgs, labels in train_ds.take(1):
    print(f"Batch shape: {imgs.shape}")
    print(f"Labels:      {labels[:8]}")
    print(f"Pixel range: {imgs.numpy().min():.2f} – {imgs.numpy().max():.2f}")
"""

'\ntrain_ds = make_dataset(train_df, shuffle=True)\n\nfor imgs, labels in train_ds.take(1):\n    print(f"Batch shape: {imgs.shape}")\n    print(f"Labels:      {labels[:8]}")\n    print(f"Pixel range: {imgs.numpy().min():.2f} – {imgs.numpy().max():.2f}")\n'

### normal dataset construction

In [ ]:
batch_size = 32
img_size = 10

def load_image(file_path):
    img = plt.imread(file_path)
    return img

def parse_sample(file_path, label, L):
    img = tf.py_function(
        func=lambda p: plt.imread(p.numpy().decode()).astype(np.float32),
        inp=[file_path],
        Tout=tf.float32
    )
    img.set_shape([L, L])      # L is already known, use it directly
    return img, label

def load_ds(df, dir, L):
    selected = df[df["L"] == L]
    paths  = (dir + "/" + selected["filepath"]).values
    labels = selected["label"].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.shuffle(buffer_size=len(selected), seed=42)
    ds = ds.map(lambda path, label: parse_sample(path, label, L))
    ds = ds.batch(32)
    #ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

sanity check

In [9]:
df = pd.read_csv(CSV_PATH)
ds = load_ds(df, DATA_DIR, 10)
for element in ds:
    print(element)

W0000 00:00:1778064674.726407 3084794 local_rendezvous.cc:412] Local rendezvous is aborting with status: INVALID_ARGUMENT: Incompatible shapes at component 0: expected [?,10,10] but got [32,200,200,4].


InvalidArgumentError: {{function_node __wrapped__IteratorGetNext_output_types_2_device_/job:localhost/replica:0/task:0/device:CPU:0}} Incompatible shapes at component 0: expected [?,10,10] but got [32,200,200,4]. [Op:IteratorGetNext] name: 

### Build model

In [8]:
x = tf.keras.Input((100, 1))
#y = tf.keras.layers.Flatten()(x)
y = tf.keras.layers.Dense(100, activation='sigmoid')(x)
z = tf.keras.layers.Dense(1, activation='sigmoid')(y)
model = tf.keras.Model(inputs=x, outputs=z)
model.compile(optimizer='adam', loss='mse')
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 100, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100, 100)       │           200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100, 1)         │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 301 (1.18 KB)

 Trainable params: 301 (1.18 KB)

 Non-trainable params: 0 (0.00 B)

In [49]:
from sklearn.model_selection import train_test_split

# skip this block if your CSV already has a "val" split and use val_ds directly
train_df, val_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df["label"]
)

train_ds = make_dataset(train_df, shuffle=True)
val_ds   = make_dataset(val_df,   shuffle=False)

In [50]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50
)

Epoch 1/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.1874 - val_loss: 0.1379
Epoch 2/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.1159 - val_loss: 0.1019
Epoch 3/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 0.0884 - val_loss: 0.0815
Epoch 4/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 0.0703 - val_loss: 0.0666
Epoch 5/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 0.0570 - val_loss: 0.0559
Epoch 6/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.0476 - val_loss: 0.0499
Epoch 7/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.0409 - val_loss: 0.0438
Epoch 8/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.0361 - val_loss: 0.0395
Epoch 9/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.0326 - val_loss: 0.0373
Epoch 10/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.0296 - val_loss: 0.0347
Epoch 11/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - loss: 0.0275 - val_loss: 0.0332
Epoch 12/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 

In [52]:
model.evaluate(train_ds, batch_size=100)

188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - loss: 0.0115


0.01149583887308836